In [16]:
import os
import shutil
from google.colab import drive

# Ensure the mount point is clean before attempting to mount
# This addresses the 'Mountpoint must not already contain files' error.
if os.path.exists('/content/drive'):
    if os.path.isdir('/content/drive'):
        print("Clearing existing /content/drive directory...")
        try:
            shutil.rmtree('/content/drive')
        except OSError as e:
            print(f"Warning: Could not remove /content/drive: {e}. You might need to restart the runtime if issues persist.")

# Create a fresh, empty directory to serve as the mount point
os.makedirs('/content/drive', exist_ok=True)

drive.mount('/content/drive', force_remount=True)

Clearing existing /content/drive directory...
Mounted at /content/drive


# Silero Voice Activity Detection (VAD)

### Purpose
This notebook detects speech and non-speech regions in recorded meeting audio using Silero VAD.

### Pipeline Position
Recorded Audio/Video → Audio Preprocessing → **Silero VAD** → Whisper ASR

### Expected Output
Speech segments with start and end timestamps that will be used by the downstream ASR and speaker diarization stages.

In [1]:
# ============================================================
# CELL 3: SET PROJECT PATHS
# Purpose:
# Define the main project directory and the directories used
# for input meeting audio and VAD output files.
# ============================================================

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/MTechIndProj/MoM_Project"
)

DATA_DIR = PROJECT_DIR / "data"
VAD_DIR = DATA_DIR / "vad"

# Create VAD output directory if it does not already exist
VAD_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("VAD output directory:", VAD_DIR)

Project directory: /content/drive/MyDrive/MTechIndProj/MoM_Project
VAD output directory: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/vad


In [3]:
# ============================================================
# CELL 4: VERIFY GPU AVAILABILITY
# Purpose:
# Check whether the Colab runtime has a CUDA-enabled GPU.
# GPU acceleration will be useful for the later WhisperX
# and Pyannote stages of the project.
# ============================================================

import torch

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU device      :", torch.cuda.get_device_name(0))
else:
    print("GPU not available - running on CPU")

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU device      : Tesla T4


In [4]:
# ============================================================
# CELL 5: INSTALL SILERO VAD DEPENDENCIES
# Purpose:
# Install the libraries required to load and run the
# Silero Voice Activity Detection (VAD) model.
# ============================================================

!pip install -q silero-vad soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 36.2 MB/s eta 0:00:00


In [5]:
# ============================================================
# CELL 6: IMPORT AND VERIFY SILERO VAD
# Purpose:
# Import the Silero VAD library and verify that the model
# can be loaded successfully in the current Colab environment.
# ============================================================

import torch
import soundfile as sf
from silero_vad import load_silero_vad

print("Silero VAD library imported successfully.")

# Select GPU when available; otherwise use CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("VAD device:", DEVICE)

Silero VAD library imported successfully.
VAD device: cuda


In [6]:
# ============================================================
# CELL 7: LOAD SILERO VAD MODEL
# Purpose:
# Load the pretrained Silero VAD model and move it to the
# available GPU for speech activity detection.
# ============================================================

print("Loading Silero VAD model...")

vad_model = load_silero_vad()

# Move model to the selected device
vad_model = vad_model.to(DEVICE)

print("Silero VAD model loaded successfully.")
print("Model device:", DEVICE)

Loading Silero VAD model...
Silero VAD model loaded successfully.
Model device: cuda


In [17]:
# ============================================================
# CELL 8: SELECT AND INSPECT INPUT MEETING AUDIO
# Purpose:
# Define the AMI meeting recording used for the first Silero
# VAD test and inspect its audio properties.
# ============================================================

# Path to the first AMI meeting audio used for testing
INPUT_AUDIO = (
    DATA_DIR
    / "raw"
    / "ami"
    / "ES2004a"
    / "audio"
    / "ES2004a.Mix-Headset.wav"
)

print("Input audio:", INPUT_AUDIO)
print("File exists:", INPUT_AUDIO.exists())

Input audio: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/audio/ES2004a.Mix-Headset.wav
File exists: True


In [18]:
# ============================================================
# CELL 8A: INSPECT AUDIO PROPERTIES
# Purpose:
# Check the sample rate, number of channels, number of samples,
# and duration of the selected meeting recording.
# ============================================================

import soundfile as sf

audio_info = sf.info(str(INPUT_AUDIO))

print("Audio format      :", audio_info.format)
print("Sample rate       :", audio_info.samplerate, "Hz")
print("Channels          :", audio_info.channels)
print("Duration          :", round(audio_info.duration / 60, 2), "minutes")
print("Total samples     :", audio_info.frames)

Audio format      : WAV
Sample rate       : 16000 Hz
Channels          : 1
Duration          : 17.49 minutes
Total samples     : 16789675


In [19]:
# ============================================================
# CELL 9: LOAD AUDIO FOR VOICE ACTIVITY DETECTION
# Purpose:
# Load the selected AMI meeting audio as a waveform and
# prepare it for Silero VAD processing.
# ============================================================

import torch
import torchaudio

# Load the meeting audio
waveform, sample_rate = torchaudio.load(str(INPUT_AUDIO))

print("Waveform shape :", waveform.shape)
print("Sample rate    :", sample_rate, "Hz")
print("Audio duration :", round(waveform.shape[1] / sample_rate / 60, 2), "minutes")

Waveform shape : torch.Size([1, 16789675])
Sample rate    : 16000 Hz
Audio duration : 17.49 minutes


In [20]:
# ============================================================
# CELL 10: RUN SILERO VOICE ACTIVITY DETECTION
# Purpose:
# Detect the portions of the meeting audio that contain speech.
# Silero VAD identifies speech start and end timestamps while
# ignoring silence and other non-speech regions.
# ============================================================

from silero_vad import get_speech_timestamps

# Move the mono waveform to the selected device
vad_waveform = waveform.squeeze(0).to(DEVICE)

print("Running Silero VAD...")
print("This may take a little time for the 17.49-minute recording.")

speech_timestamps = get_speech_timestamps(
    vad_waveform,
    vad_model,
    sampling_rate=sample_rate,
    threshold=0.5,
    min_speech_duration_ms=250,
    min_silence_duration_ms=100,
    speech_pad_ms=30
)

print("\nVAD completed successfully.")
print("Number of speech segments:", len(speech_timestamps))

Running Silero VAD...
This may take a little time for the 17.49-minute recording.

VAD completed successfully.
Number of speech segments: 281


Silero VAD completed successfully and detected 281 speech segments in the 17.49-minute meeting.

In [21]:
# ============================================================
# CELL 11: INSPECT VAD SPEECH SEGMENTS
# Purpose:
# Display the first few speech segments detected by Silero VAD.
# This helps verify that the detected timestamps are reasonable
# before saving the VAD results for downstream processing.
# ============================================================

def format_timestamp(seconds):
    """Convert seconds into HH:MM:SS.ss format."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60
    return f"{hours:02d}:{minutes:02d}:{secs:05.2f}"


print("First 10 detected speech segments:\n")

for i, segment in enumerate(speech_timestamps[:10], start=1):
    start_sec = segment["start"] / sample_rate
    end_sec = segment["end"] / sample_rate

    print(
        f"Segment {i:02d}: "
        f"{format_timestamp(start_sec)} → "
        f"{format_timestamp(end_sec)} "
        f"(Duration: {end_sec - start_sec:.2f} sec)"
    )

First 10 detected speech segments:

Segment 01: 00:00:12.48 → 00:00:14.65 (Duration: 2.17 sec)
Segment 02: 00:00:22.59 → 00:00:23.84 (Duration: 1.24 sec)
Segment 03: 00:00:25.19 → 00:00:26.65 (Duration: 1.47 sec)
Segment 04: 00:00:29.35 → 00:00:30.49 (Duration: 1.15 sec)
Segment 05: 00:00:31.27 → 00:00:31.61 (Duration: 0.35 sec)
Segment 06: 00:00:31.75 → 00:00:32.35 (Duration: 0.60 sec)
Segment 07: 00:01:22.63 → 00:01:23.77 (Duration: 1.15 sec)
Segment 08: 00:01:24.55 → 00:01:25.69 (Duration: 1.15 sec)
Segment 09: 00:01:26.59 → 00:01:27.36 (Duration: 0.76 sec)
Segment 10: 00:01:27.91 → 00:01:30.17 (Duration: 2.27 sec)


The VAD output is behaving as expected. It is detecting short speech regions and separating them when there is sufficient silence between them.

For example:

Segment 01 → ~2.17 seconds of speech
Segment 05 → ~0.35 seconds
Segment 06 → ~0.60 seconds
The gap between Segments 06 and 07 is large, so VAD correctly treats them as separate speech regions.

We can now save all 281 detected segments with both sample positions and human-readable timestamps.

In [22]:
# ============================================================
# CELL 12: SAVE SILERO VAD RESULTS
# Purpose:
# Save all detected speech segments with their start/end
# timestamps so they can be reused by the downstream
# ASR and speaker diarization stages.
# ============================================================

import json

vad_results = []

for i, segment in enumerate(speech_timestamps, start=1):
    start_sample = segment["start"]
    end_sample = segment["end"]

    start_seconds = start_sample / sample_rate
    end_seconds = end_sample / sample_rate

    vad_results.append({
        "segment_id": i,
        "start_sample": start_sample,
        "end_sample": end_sample,
        "start_seconds": round(start_seconds, 3),
        "end_seconds": round(end_seconds, 3),
        "start_timestamp": format_timestamp(start_seconds),
        "end_timestamp": format_timestamp(end_seconds),
        "duration_seconds": round(end_seconds - start_seconds, 3)
    })

# Define output file
VAD_OUTPUT_FILE = VAD_DIR / "ES2004a_silero_vad.json"

# Save results
with open(VAD_OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(vad_results, f, indent=4)

print("VAD results saved successfully.")
print("Output file:", VAD_OUTPUT_FILE)
print("Total speech segments saved:", len(vad_results))

VAD results saved successfully.
Output file: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/vad/ES2004a_silero_vad.json
Total speech segments saved: 281


In [23]:
# ============================================================
# CELL 13: VAD SUMMARY AND QUALITY CHECK
# Purpose:
# Calculate basic statistics from the detected speech segments.
# These values help verify the VAD output and document the
# preprocessing stage of the project.
# ============================================================

import numpy as np

# Extract speech durations
speech_durations = [
    segment["duration_seconds"]
    for segment in vad_results
]

# Calculate total speech duration
total_speech_seconds = sum(speech_durations)

# Calculate percentage of the meeting containing speech
meeting_duration_seconds = waveform.shape[1] / sample_rate
speech_percentage = (
    total_speech_seconds / meeting_duration_seconds
) * 100

# Calculate silence percentage
silence_percentage = 100 - speech_percentage

print("========== SILERO VAD SUMMARY ==========")
print("Meeting duration       :", round(meeting_duration_seconds / 60, 2), "minutes")
print("Speech segments        :", len(vad_results))
print("Total speech duration  :", round(total_speech_seconds / 60, 2), "minutes")
print("Total silence duration :", round(
    (meeting_duration_seconds - total_speech_seconds) / 60, 2
), "minutes")
print("Speech percentage      :", round(speech_percentage, 2), "%")
print("Silence percentage     :", round(silence_percentage, 2), "%")
print("Average segment length :", round(np.mean(speech_durations), 2), "seconds")
print("Shortest segment       :", round(np.min(speech_durations), 2), "seconds")
print("Longest segment        :", round(np.max(speech_durations), 2), "seconds")

========== SILERO VAD SUMMARY ==========
Meeting duration       : 17.49 minutes
Speech segments        : 281
Total speech duration  : 11.03 minutes
Total silence duration : 6.46 minutes
Speech percentage      : 63.08 %
Silence percentage     : 36.92 %
Average segment length : 2.36 seconds
Shortest segment       : 0.35 seconds
Longest segment        : 14.78 seconds
